# 01 Data Acquisition
---

Pull elliptic-curve metadata from the LMFDB Postgres mirror, compute the first 1000 Frobenius traces $a_{p_n}(E)$ with PARI, and cache to parquet.

One representative per isogeny class is kept (`lmfdb_number = 1`), matching the paper, since $a_p$ and rank are isogeny-class invariants.

In [1]:
%load_ext autoreload
%autoreload 2
from functools import partial
from pathlib import Path

from murmurations.data import build_balanced_dataset, load_or_build
from murmurations.lmfdb import count_curves

DATA = Path("..") / "data"

### Check the isogeny-class counts against HLOP

Example 1 reports `#E0[7500,10000] = 4328` and `#E1[7500,10000] = 5194`.

In [2]:
for r, expected in [(0, 4328), (1, 5194)]:
    print(f"rank {r}: {count_curves(7500, 10000, rank=r)}  (paper: {expected})")

rank 0: 4328  (paper: 4328)
rank 1: 5194  (paper: 5194)


### Build the primary dataset: ranks {0,1,2}, conductor [7500, 10000]
First run pulls from LMFDB and computes $a_p$ (a few minutes); afterwards it loads instantly from parquet.

In [3]:
ec_7500_10000 = load_or_build(
    DATA / "ec_7500_10000_r012_1000ap.parquet",
    conductor_min=7500,
    conductor_max=10000,
    ranks=[0, 1, 2],
    num_primes=1000,
)
ec_7500_10000[["lmfdb_label", "conductor", "rank"]].groupby("rank").size()

rank
0    4328
1    5194
2     771
dtype: int64

In [4]:
ec_7500_10000.iloc[:3, :10]  # a_p feature columns are ap0001 ... ap1000

,lmfdb_label,lmfdb_iso,conductor,rank,torsion,class_size,ap0001,ap0002,ap0003,ap0004
0,7501.a1,7501.a,7501,0,1,1,1,2,-2,4
1,7502.a1,7502.a,7502,1,1,3,-1,-2,0,1
2,7502.b1,7502.b,7502,0,2,4,-1,0,-2,0


### A larger sample for classification (conductor up to 1e5)
The paper draws balanced random samples of 20k curves/rank for Table 1. The following cells take ~10 minutes to execute.

In [ ]:
ec_1_100000 = load_or_build(
    DATA / "ec_1_100000_balanced20k_1000ap.parquet",
    builder=partial(
        build_balanced_dataset,
        conductor_min=1,
        conductor_max=100_000,
        ranks=[0, 1, 2, 3],
        n_per_rank=20_000,
        num_primes=1000,
        seed=0.42,  # Postgres setseed() arg, float in [-1, 1] -- not a np seed
    ),
)
ec_1_100000.groupby("rank").size()

rank
0    20000
1    20000
2    20000
3      531
dtype: int64

In [6]:
n2 = count_curves(10_000, 40_000, rank=2)
print("rank 2 available in [10000, 40000]:", n2)
n_per = min(12_000, n2)

ec_10000_40000 = load_or_build(
    DATA / "ec_10000_40000_balanced_1000ap.parquet",
    builder=partial(
        build_balanced_dataset,
        conductor_min=10_000,
        conductor_max=40_000,
        ranks=[0, 1, 2],
        n_per_rank=n_per,
        num_primes=1000,
        seed=0.42,
    ),
)
ec_10000_40000.groupby("rank").size()

rank 2 available in [10000, 40000]: 12190


rank
0    12000
1    12000
2    12000
dtype: int64

In [7]:
ec_1_10000 = load_or_build(
    DATA / "ec_1_10000_r0123_1000ap.parquet",
    builder=partial(
        build_balanced_dataset,
        conductor_min=1,
        conductor_max=10_000,
        ranks=[0, 1, 2, 3],
        n_per_rank=2_000,
        num_primes=1000,
        seed=0.42,
    ),
)
ec_1_10000.groupby("rank").size()

rank
0    2000
1    2000
2    1969
3       1
dtype: int64